# Baseline — DummyClassifier

Dataset: IBM Telco Customer Churn (~7.000 registros, 20 features, target `Churn`)

Estratégias:
- `most_frequent`: sempre prediz a classe majoritária (No Churn)
- `stratified`: prediz aleatoriamente respeitando a proporção do target

## 1. Setup

In [ ]:
from pathlib import Path

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold

RANDOM_SEED = 42
TARGET = "Churn"
N_SPLITS = 5
PROJECT_ROOT = Path.cwd().parent

## 2. Carregar e limpar

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(DATA_PATH)

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df[TARGET] = df[TARGET].map({"Yes": 1, "No": 0})
df = df.drop(columns=["customerID"])

print(f"Shape: {df.shape}")
print(f"Distribuição do target:\n{df[TARGET].value_counts(normalize=True)}")

## 3. Validação cruzada (StratifiedKFold k=5)

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)
print(f"Dataset: {X.shape[0]} amostras | Validação cruzada: {N_SPLITS} folds")

## 4. Dummy — `most_frequent`

In [ ]:
dummy_mf = DummyClassifier(strategy="most_frequent", random_state=RANDOM_SEED)
mf_metrics = {k: [] for k in ["accuracy", "roc_auc", "recall", "precision", "f1"]}

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    m = clone(dummy_mf)
    m.fit(X_tr, y_tr)
    y_pred = m.predict(X_val)
    y_prob = m.predict_proba(X_val)[:, 1]

    mf_metrics["accuracy"].append(accuracy_score(y_val, y_pred))
    mf_metrics["roc_auc"].append(roc_auc_score(y_val, y_prob))
    mf_metrics["recall"].append(recall_score(y_val, y_pred, zero_division=0))
    mf_metrics["precision"].append(precision_score(y_val, y_pred, zero_division=0))
    mf_metrics["f1"].append(f1_score(y_val, y_pred, zero_division=0))

mf_means = {k: float(np.mean(v)) for k, v in mf_metrics.items()}
mf_stds = {k: float(np.std(v)) for k, v in mf_metrics.items()}

print("Resultados (média ± std):")
for metric in ["accuracy", "roc_auc", "recall", "precision", "f1"]:
    print(f"  {metric:12s}: {mf_means[metric]:.4f} ± {mf_stds[metric]:.4f}")

## 5. Dummy — `stratified`

In [ ]:
dummy_st = DummyClassifier(strategy="stratified", random_state=RANDOM_SEED)
st_metrics = {k: [] for k in ["accuracy", "roc_auc", "recall", "precision", "f1"]}

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    m = clone(dummy_st)
    m.fit(X_tr, y_tr)
    y_pred = m.predict(X_val)
    y_prob = m.predict_proba(X_val)[:, 1]

    st_metrics["accuracy"].append(accuracy_score(y_val, y_pred))
    st_metrics["roc_auc"].append(roc_auc_score(y_val, y_prob))
    st_metrics["recall"].append(recall_score(y_val, y_pred, zero_division=0))
    st_metrics["precision"].append(precision_score(y_val, y_pred, zero_division=0))
    st_metrics["f1"].append(f1_score(y_val, y_pred, zero_division=0))

st_means = {k: float(np.mean(v)) for k, v in st_metrics.items()}
st_stds = {k: float(np.std(v)) for k, v in st_metrics.items()}

print("Resultados (média ± std):")
for metric in ["accuracy", "roc_auc", "recall", "precision", "f1"]:
    print(f"  {metric:12s}: {st_means[metric]:.4f} ± {st_stds[metric]:.4f}")

## 6. Registrar no MLflow

In [ ]:
mlflow.set_tracking_uri(f"sqlite:///{PROJECT_ROOT / 'mlflow.db'}")
mlflow.set_experiment("churn-baselines")

for name, means, stds, base_model in [
    ("dummy_most_frequent", mf_means, mf_stds, dummy_mf),
    ("dummy_stratified", st_means, st_stds, dummy_st),
]:
    with mlflow.start_run(run_name=name):
        mlflow.log_params({
            "model": "DummyClassifier",
            "strategy": name.replace("dummy_", ""),
            "random_seed": RANDOM_SEED,
            "cv_folds": N_SPLITS,
        })
        mlflow.log_metrics(means)
        mlflow.log_metrics({f"{k}_std": v for k, v in stds.items()})
        final = clone(base_model).fit(X, y)
        mlflow.sklearn.log_model(final, "model")
        print(f"{name} → run_id: {mlflow.active_run().info.run_id}")